In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
df = pd.read_csv("loanapproval.csv")

In [4]:
df.head()

,applicant_id,age,gender,marital_status,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count,employment_status,loan_approved
0,1,59,Male,Divorced,100073,7169,793,1,1,Unemployed,1
1,2,49,Male,Married,112197,23556,789,0,2,Employed,1
2,3,35,Male,Divorced,84429,27052,372,1,4,Unemployed,0
3,4,63,Female,Single,124195,11313,808,3,4,Self-employed,1
4,5,28,Female,Married,81627,13315,689,0,1,Unemployed,1


In [5]:
df.shape

(1000, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   applicant_id          1000 non-null   int64 
 1   age                   1000 non-null   int64 
 2   gender                1000 non-null   object
 3   marital_status        1000 non-null   object
 4   annual_income         1000 non-null   int64 
 5   loan_amount           1000 non-null   int64 
 6   credit_score          1000 non-null   int64 
 7   num_dependents        1000 non-null   int64 
 8   existing_loans_count  1000 non-null   int64 
 9   employment_status     1000 non-null   object
 10  loan_approved         1000 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 86.1+ KB


In [7]:
df.isnull().sum()

applicant_id            0
age                     0
gender                  0
marital_status          0
annual_income           0
loan_amount             0
credit_score            0
num_dependents          0
existing_loans_count    0
employment_status       0
loan_approved           0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df = df.drop('applicant_id', axis=1)

In [10]:
df.head()

,age,gender,marital_status,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count,employment_status,loan_approved
0,59,Male,Divorced,100073,7169,793,1,1,Unemployed,1
1,49,Male,Married,112197,23556,789,0,2,Employed,1
2,35,Male,Divorced,84429,27052,372,1,4,Unemployed,0
3,63,Female,Single,124195,11313,808,3,4,Self-employed,1
4,28,Female,Married,81627,13315,689,0,1,Unemployed,1


In [11]:
df = pd.get_dummies(df,columns = ['gender','marital_status','employment_status'],drop_first=True)

In [12]:
df = df.astype(int)

In [13]:
df['loan_approved'].value_counts()

loan_approved
1    729
0    271
Name: count, dtype: int64

In [14]:
df.head()

,age,annual_income,loan_amount,credit_score,num_dependents,existing_loans_count,loan_approved,gender_Male,marital_status_Married,marital_status_Single,employment_status_Self-employed,employment_status_Unemployed
0,59,100073,7169,793,1,1,1,1,0,0,0,1
1,49,112197,23556,789,0,2,1,1,1,0,0,0
2,35,84429,27052,372,1,4,0,1,0,0,0,1
3,63,124195,11313,808,3,4,1,0,0,1,1,0
4,28,81627,13315,689,0,1,1,0,1,0,0,1


In [15]:
df.columns

Index(['age', 'annual_income', 'loan_amount', 'credit_score', 'num_dependents',
       'existing_loans_count', 'loan_approved', 'gender_Male',
       'marital_status_Married', 'marital_status_Single',
       'employment_status_Self-employed', 'employment_status_Unemployed'],
      dtype='object')

In [16]:
from sklearn.preprocessing import StandardScaler
cols = ['age', 'annual_income', 'loan_amount', 'credit_score', 'num_dependents',
       'existing_loans_count']
scaler = StandardScaler()

df[cols] = scaler.fit_transform(df[cols])

In [17]:
from scipy.stats import pearsonr

# ----------------------------------
# Pearson Correlation Calculation
# ----------------------------------

# List of features to check against target
selected_features = [
    'age', 'annual_income', 'loan_amount', 'credit_score', 'num_dependents',
       'existing_loans_count', 'loan_approved', 'gender_Male',
       'marital_status_Married', 'marital_status_Single',
       'employment_status_Self-employed', 'employment_status_Unemployed'
]

# Calculate Pearson correlation
correlations = {
    feature: pearsonr(df[feature], df['loan_approved'])[0]
    for feature in selected_features
}

# Convert to DataFrame for display
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])

# Show the results
correlation_df.sort_values(by='Pearson Correlation', ascending=False)

,Feature,Pearson Correlation
6,loan_approved,1.000000
3,credit_score,0.351167
10,employment_status_Self-employed,0.206838
1,annual_income,0.176453
8,marital_status_Married,0.062086
0,age,0.020984
7,gender_Male,0.018038
2,loan_amount,0.015355
9,marital_status_Single,-0.044192
5,existing_loans_count,-0.324236


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X = df.drop('loan_approved', axis=1)
y = df['loan_approved']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [21]:
from sklearn.linear_model import LogisticRegression

In [22]:
model_lr = LogisticRegression()
model_lr.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [23]:
y_pred_lr = model_lr.predict(X_test)

In [26]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
accuracy = accuracy_score(y_test, y_pred_lr)
print("Accuracy:", accuracy)

Accuracy: 0.87


In [27]:
cm = confusion_matrix(y_test, y_pred_lr)
print(cm)

[[ 40  15]
 [ 11 134]]


In [28]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.78      0.73      0.75        55
           1       0.90      0.92      0.91       145

    accuracy                           0.87       200
   macro avg       0.84      0.83      0.83       200
weighted avg       0.87      0.87      0.87       200



In [29]:
ridge_model = LogisticRegression(
    penalty='l2',
    C=0.1,
    solver='liblinear'
)

ridge_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [30]:
y_pred_r = ridge_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_r)
print("Accuracy:", accuracy)
cm = confusion_matrix(y_test, y_pred_r)
print(cm)
print(classification_report(y_test, y_pred_r))

Accuracy: 0.865
[[ 40  15]
 [ 12 133]]
              precision    recall  f1-score   support

           0       0.77      0.73      0.75        55
           1       0.90      0.92      0.91       145

    accuracy                           0.86       200
   macro avg       0.83      0.82      0.83       200
weighted avg       0.86      0.86      0.86       200



In [31]:
lasso_model = LogisticRegression(
    penalty='l1',
    C=0.1,
    solver='liblinear'
)

lasso_model.fit(X_train, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [32]:
y_pred_l = ridge_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_l)
print("Accuracy:", accuracy)
cm = confusion_matrix(y_test, y_pred_l)
print(cm)
print(classification_report(y_test, y_pred_l))

Accuracy: 0.865
[[ 40  15]
 [ 12 133]]
              precision    recall  f1-score   support

           0       0.77      0.73      0.75        55
           1       0.90      0.92      0.91       145

    accuracy                           0.86       200
   macro avg       0.83      0.82      0.83       200
weighted avg       0.86      0.86      0.86       200



In [33]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

Accuracy: 0.96
              precision    recall  f1-score   support

           0       1.00      0.85      0.92        55
           1       0.95      1.00      0.97       145

    accuracy                           0.96       200
   macro avg       0.97      0.93      0.95       200
weighted avg       0.96      0.96      0.96       200



In [35]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

In [36]:
print("Accuracy:", accuracy_score(y_test, gb_pred))

print(classification_report(y_test, gb_pred))

Accuracy: 0.98
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        55
           1       0.97      1.00      0.99       145

    accuracy                           0.98       200
   macro avg       0.99      0.96      0.97       200
weighted avg       0.98      0.98      0.98       200

